# 🚖 NYC Taxi Anomalous Trip Detection

## Use Case 5: Identify Anomalous Trips

### Notebook 1: Data Loading & Initial Data Quality Assessment

### Objective

The purpose of this notebook is to:

1. Load NYC Yellow Taxi trip data
2. Understand dataset structure
3. Identify missing values
4. Detect invalid records
5. Perform initial data cleaning
6. Create a clean foundation for anomaly detection

This notebook does NOT perform feature engineering or model training.
Those steps will be completed in later notebooks.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


# Step 1: Load Dataset

We begin by loading the NYC Yellow Taxi dataset.

This dataset contains detailed information about taxi trips including:

- Pickup time
- Dropoff time
- Trip distance
- Fare amount
- Tip amount
- Passenger count

Each row represents one taxi trip.

In [21]:
DATA_PATH = "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE5_Anomaly Detection/data/yellow_tripdata_2026-01.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [22]:
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(3724889, 20)


In [23]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


# Step 2: Understand Dataset Structure

Before cleaning the data, we need to understand:

- Number of rows
- Number of columns
- Data types
- Memory usage

This helps identify columns suitable for anomaly detection.

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [25]:
df.columns.tolist()


['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

# Step 3: Dataset Statistics

Summary statistics help us identify:

- Extreme values
- Unexpected values
- Distribution ranges

These observations often provide early indicators of anomalies.

In [9]:
df.describe(include="all")

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
count,3.724889e+06,3724889,3724889,2.636831e+06,3.724889e+06,2.636831e+06,2636831,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,2.636831e+06,2.636831e+06,3.724889e+06
unique,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,2634494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,1.873598e+00,2026-01-17 01:44:52.106518,2026-01-17 02:02:03.696138,1.256271e+00,6.455647e+00,5.218853e+00,NaN,1.614372e+02,1.609936e+02,8.465637e-01,2.080425e+01,1.023055e+00,4.833766e-01,2.608142e+00,4.983751e-01,9.479130e-01,2.917853e+01,2.154870e+00,1.482938e-01,5.196437e-01
min,1.000000e+00,2025-12-31 23:57:29,2025-12-31 23:57:32,0.000000e+00,0.000000e+00,1.000000e+00,NaN,1.000000e+00,1.000000e+00,0.000000e+00,-2.555200e+03,-7.500000e+00,-5.000000e-01,-8.888000e+01,-9.450000e+01,-1.000000e+00,-2.560200e+03,-2.500000e+00,-1.750000e+00,-7.500000e-01
25%,2.000000e+00,2026-01-09 17:51:59,2026-01-09 18:08:47,1.000000e+00,1.000000e+00,1.000000e+00,NaN,1.140000e+02,1.070000e+02,0.000000e+00,1.000000e+01,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.700000e+01,2.500000e+00,0.000000e+00,0.000000e+00
50%,2.000000e+00,2026-01-16 21:20:56,2026-01-16 21:36:14,1.000000e+00,1.810000e+00,1.000000e+00,NaN,1.610000e+02,1.620000e+02,1.000000e+00,1.560000e+01,0.000000e+00,5.000000e-01,2.000000e+00,0.000000e+00,1.000000e+00,2.305000e+01,2.500000e+00,0.000000e+00,7.500000e-01
75%,2.000000e+00,2026-01-24 07:25:11,2026-01-24 07:40:22,1.000000e+00,3.730000e+00,1.000000e+00,NaN,2.330000e+02,2.340000e+02,1.000000e+00,2.610000e+01,2.500000e+00,5.000000e-01,3.710000e+00,0.000000e+00,1.000000e+00,3.383000e+01,2.500000e+00,0.000000e+00,7.500000e-01
max,7.000000e+00,2026-02-01 00:45:01,2026-02-01 23:35:31,9.000000e+00,2.690975e+05,9.900000e+01,NaN,2.650000e+02,2.650000e+02,4.000000e+00,2.555200e+03,1.746000e+01,4.750000e+00,7.660000e+02,1.222200e+02,1.000000e+00,2.560200e+03,2.500000e+00,2.675000e+01,7.500000e-01


# Step 4: Check Missing Values

Missing values can negatively impact anomaly detection models.

We calculate:

- Missing count
- Missing percentage

for every column.

In [26]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": round(df.isna().mean()*100,2)
})

missing.sort_values(
    by="Missing Percentage",
    ascending=False
).head(20)

,Missing Count,Missing Percentage
passenger_count,1088058,29.21
congestion_surcharge,1088058,29.21
store_and_fwd_flag,1088058,29.21
RatecodeID,1088058,29.21
Airport_fee,1088058,29.21
tpep_dropoff_datetime,0,0.00
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00


['tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'tip_amount',
 'total_amount']

# Step 5: Remove Duplicate Records

Duplicate trips may artificially influence model behavior.

We remove exact duplicate rows.

In [11]:
initial_rows = len(df)

df = df.drop_duplicates()

final_rows = len(df)

print(f"Rows Before : {initial_rows:,}")
print(f"Rows After  : {final_rows:,}")
print(f"Duplicates Removed : {initial_rows-final_rows:,}")

Rows Before : 3,724,889
Rows After  : 3,724,889
Duplicates Removed : 0


# Step 6: Select Relevant Columns

For anomaly detection we only keep columns that contribute
to trip behavior analysis.

This reduces memory usage and simplifies downstream processing.

In [ ]:
columns_needed = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "RatecodeID"
]

df = df[columns_needed]

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,Airport_fee,total_amount,congestion_surcharge,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,0.0,15.86,2.5,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,0.0,13.65,2.5,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,0.0,18.95,2.5,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,0.0,55.56,2.5,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,0.0,23.10,2.5,0.75


# Step 7: Convert Datetime Columns

Datetime calculations are required later for:

- Trip duration
- Speed estimation
- Temporal anomaly analysis

Therefore datetime fields must be converted properly.

In [28]:
df["tpep_pickup_datetime"] = pd.to_datetime(
    df["tpep_pickup_datetime"]
)

df["tpep_dropoff_datetime"] = pd.to_datetime(
    df["tpep_dropoff_datetime"]
)

# Step 8: Basic Data Quality Checks

We identify impossible records that indicate
data quality issues.

Examples:

- Negative fare
- Negative distance
- Negative total amount

These records are invalid and will be removed.

In [29]:
invalid_distance = (df["trip_distance"] < 0).sum()

invalid_fare = (df["fare_amount"] < 0).sum()

invalid_total = (df["total_amount"] < 0).sum()

print("Invalid Distance :", invalid_distance)
print("Invalid Fare     :", invalid_fare)
print("Invalid Total    :", invalid_total)

Invalid Distance : 0
Invalid Fare     : 39463
Invalid Total    : 39984


# Step 9: Remove Invalid Records

These observations are not useful for anomaly modeling
because they represent obvious data-entry problems.

Examples:

- Distance ≤ 0
- Fare ≤ 0
- Total Amount ≤ 0

Such records are removed.

In [30]:
df = df[
    (df["trip_distance"] > 0)
    & (df["fare_amount"] > 0)
    & (df["total_amount"] > 0)
]

In [32]:
print("Shape After Cleaning:")
print(df.shape)

Shape After Cleaning:
(3560826, 19)


# Step 10: Verify Date Consistency

Dropoff time should always occur after pickup time.

Trips violating this condition indicate data corruption.

In [33]:
invalid_time = (
    df["tpep_dropoff_datetime"]
    <
    df["tpep_pickup_datetime"]
).sum()

print("Invalid Time Records:", invalid_time)

Invalid Time Records: 1


# Step 11: Final Data Quality Summary

We generate a final summary after cleaning.

This becomes the baseline dataset for feature engineering.

In [34]:
print("="*50)

print("Final Dataset Shape")
print(df.shape)

print("="*50)

print("Missing Values")

print(df.isna().sum())

print("="*50)

Final Dataset Shape
(3560826, 19)
Missing Values
VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          994749
trip_distance                 0
RatecodeID               994749
store_and_fwd_flag       994749
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
Airport_fee              994749
total_amount                  0
congestion_surcharge     994749
cbd_congestion_fee            0
dtype: int64


# Step 12: Save Clean Dataset

The cleaned dataset will be used in Notebook 2
for feature engineering and exploratory analysis.

In [ ]:
OUTPUT_PATH = "../data/processed/validate_taxi_data.parquet"

df.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Clean Dataset Saved Successfully")